# 🎤 Full Voice Confidence Deep Learning Model (CNN + BiLSTM)
This notebook trains a deep learning model to predict **voice confidence scores** from audio.
It uses **RAVDESS + CREMA-D** emotion datasets and a **CNN + Bidirectional LSTM** architecture.

> **CRITICAL FIRST STEP:** Go to **Runtime > Change runtime type** and select **T4 GPU**.

In [ ]:
# ============================================================
# CELL 1: Install all required Python packages
# - torch / torchaudio  → deep learning framework + audio loading
# - librosa             → audio feature extraction (MFCCs, pitch, etc.)
# - soundfile           → reading audio files
# - praat-parselmouth   → Praat speech analysis (jitter, shimmer, HNR)
#                         NOTE: PyPI name is 'praat-parselmouth' (not parselmouth-praat)
# - scikit-learn        → train/test split, label encoding, metrics
# - matplotlib/seaborn  → plotting graphs
# - tqdm                → progress bars during loops
# - google-generativeai → Gemini API (used in inference cell)
# - openai-whisper      → speech-to-text transcription
# ============================================================
!pip install -q torch torchaudio librosa soundfile
!pip install -q praat-parselmouth scikit-learn
!pip install -q matplotlib seaborn tqdm pandas numpy
!pip install -q google-generativeai openai-whisper
print("✅ All packages installed!")


In [ ]:
# ============================================================
# CELL 2: Import all libraries and set global configuration
# - os, glob, random    → file system traversal and randomness
# - numpy / pandas      → numerical arrays and dataframe handling
# - librosa             → audio loading and acoustic features
# - torch (nn, optim)   → building and training neural networks
# - torchaudio (T)      → audio transforms (MelSpectrogram, Resample)
# - parselmouth / praat → prosodic feature extraction via Praat engine
# - sklearn             → splitting data, encoding labels, reporting metrics
# - matplotlib / sns    → visualization
# - tqdm                → loop progress bars
#
# SEED = 42 ensures all random operations are reproducible
# device = 'cuda' if GPU is available, else falls back to 'cpu'
# ============================================================
import os
import glob
import random
import numpy as np
import pandas as pd
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import parselmouth
from parselmouth.praat import call
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")


In [ ]:
# ============================================================
# CELL 3: Download the RAVDESS Dataset
# - RAVDESS = Ryerson Audio-Visual Database of Emotional Speech and Song
# - Contains ~1440 .wav files from 24 professional actors
# - Emotions: neutral, calm, happy, sad, angry, fearful, disgust, surprised
# - Steps:
#   1. Create the target folder /content/data/ravdess/
#   2. Download the zip (~500 MB) from Zenodo using wget
#   3. Extract the zip into the folder
#   4. Delete the zip to free space
# ============================================================
import zipfile
print("📥 Downloading RAVDESS...")
!mkdir -p /content/data/ravdess
!wget -q --show-progress "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip" -O /content/data/ravdess/ravdess.zip
print("📦 Extracting...")
with zipfile.ZipFile('/content/data/ravdess/ravdess.zip', 'r') as z:
    z.extractall('/content/data/ravdess/')
os.remove('/content/data/ravdess/ravdess.zip')
print("✅ RAVDESS ready!")


In [ ]:
# ============================================================
# CELL 4: Download the CREMA-D Dataset
# - CREMA-D = Crowd Sourced Emotional Multimodal Actors Dataset
# - Contains ~7442 .wav files from 91 diverse actors
# - Emotions: Angry, Disgust, Fear, Happy, Neutral, Sad
# - We git clone the GitHub repo (depth=1 for speed, no full history)
# - Audio files will be at: /content/data/cremad/repo/AudioWAV/
# - Takes ~3 minutes depending on Colab network speed
# ============================================================
print("📥 Downloading CREMA-D (this takes ~3 min)...")
!mkdir -p /content/data/cremad
!git clone --depth=1 https://github.com/CheyneyComputerScience/CREMA-D.git /content/data/cremad/repo
print("✅ CREMA-D ready!")


## Setup Labels and Prosodic Features (Part 1)

In [ ]:
# ============================================================
# CELL 5: Parse datasets and derive confidence score labels
#
# --- RAVDESS filename format: 03-01-04-01-02-01-12.wav ---
#   Position 3 = emotion code (e.g. '04' = sad)
#   Position 4 = intensity ('01'=normal, '02'=strong)
#   parse_ravdess() reads all .wav files, decodes filename, maps to emotion
#
# --- CREMA-D filename format: 1001_DFA_ANG_XX.wav ---
#   Position 3 = emotion code (e.g. 'ANG' = angry)
#   parse_cremad() reads all .wav files and maps to emotion name
#
# --- Confidence Score Derivation ---
#   Since we have no ground-truth confidence labels, we derive them
#   from emotion psychology research (e.g., 'angry' voice = high confidence)
#   EMOTION_CONFIDENCE  → base confidence score per emotion
#   EMOTION_PITCH_STABILITY → how stable the pitch is per emotion
#   EMOTION_FLUENCY     → speech fluency/smoothness per emotion
#   derive_scores() adds small random noise to make scores realistic
#   Strong intensity gets a +0.08 bonus to confidence & fluency
#
# --- build_dataset() ---
#   1. Parses both datasets and merges them
#   2. Filters to 6 emotions: neutral, happy, sad, angry, fearful, disgust
#   3. Derives confidence/pitch/fluency/energy scores for each sample
#   4. Encodes emotion strings as integer labels (0,1,2,...)
# ============================================================
EMOTION_MAP_RAVDESS = {'01':'neutral','02':'neutral','03':'happy','04':'sad','05':'angry','06':'fearful','07':'disgust','08':'surprised'}
EMOTION_MAP_CREMAD = {'ANG':'angry','DIS':'disgust','FEA':'fearful','HAP':'happy','NEU':'neutral','SAD':'sad'}
INTENSITY_MAP = {'01': 'normal', '02': 'strong'}

def parse_ravdess(base_path="/content/data/ravdess"):
    records = []
    for fp in glob.glob(os.path.join(base_path, "**/*.wav"), recursive=True):
        parts = os.path.basename(fp).replace('.wav','').split('-')
        if len(parts) != 7: continue
        emotion = EMOTION_MAP_RAVDESS.get(parts[2])
        intensity = INTENSITY_MAP.get(parts[3], 'normal')
        actor = int(parts[6])
        if emotion: records.append({'filepath':fp, 'emotion':emotion, 'intensity':intensity, 'dataset':'ravdess'})
    df = pd.DataFrame(records)
    print(f"✅ RAVDESS: {len(df)} files")
    return df

def parse_cremad(base_path="/content/data/cremad/repo/AudioWAV"):
    records = []
    for fp in glob.glob(os.path.join(base_path, "*.wav")):
        parts = os.path.basename(fp).replace('.wav','').split('_')
        if len(parts) < 3: continue
        emotion = EMOTION_MAP_CREMAD.get(parts[2])
        if emotion: records.append({'filepath':fp, 'emotion':emotion, 'intensity':'normal', 'dataset':'cremad'})
    df = pd.DataFrame(records)
    print(f"✅ CREMA-D: {len(df)} files")
    return df

EMOTION_CONFIDENCE = {'angry':0.78, 'happy':0.82, 'neutral':0.65, 'sad':0.28, 'fearful':0.18, 'disgust':0.55, 'surprised':0.60}
EMOTION_PITCH_STABILITY = {'angry':0.50, 'happy':0.72, 'neutral':0.80, 'sad':0.60, 'fearful':0.35, 'disgust':0.58, 'surprised':0.45}
EMOTION_FLUENCY = {'angry':0.70, 'happy':0.80, 'neutral':0.75, 'sad':0.45, 'fearful':0.35, 'disgust':0.60, 'surprised':0.55}

def derive_scores(row, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    noise = lambda: rng.uniform(-0.06, 0.06)
    intensity_bonus = 0.08 if row['intensity'] == 'strong' else 0.0
    conf  = np.clip(EMOTION_CONFIDENCE.get(row['emotion'], 0.5)      + intensity_bonus + noise(), 0, 1)
    pitch = np.clip(EMOTION_PITCH_STABILITY.get(row['emotion'], 0.5) + noise(), 0, 1)
    flu   = np.clip(EMOTION_FLUENCY.get(row['emotion'], 0.5)         + intensity_bonus*0.5 + noise(), 0, 1)
    eng   = np.clip(conf * 0.9 + noise(), 0, 1)
    return conf, pitch, flu, eng

def build_dataset():
    rng = np.random.default_rng(SEED)
    df_r = parse_ravdess()
    df_c = parse_cremad()
    df = pd.concat([df_r, df_c], ignore_index=True)
    keep = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust']
    df = df[df['emotion'].isin(keep)].reset_index(drop=True)
    scores = df.apply(lambda row: derive_scores(row, rng), axis=1, result_type='expand')
    scores.columns = ['confidence_score', 'pitch_score', 'fluency_score', 'energy_score']
    df = pd.concat([df, scores], axis=1)
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['emotion'])
    return df, le

df, label_encoder = build_dataset()


In [ ]:
# ============================================================
# CELL 6: Extract Prosodic (Acoustic) Features from each audio file
#
# PROSODIC_COLS = 13 hand-crafted acoustic features per file:
#   pitch_mean       → average fundamental frequency (F0) in Hz
#   pitch_std        → variation in pitch (monotone vs expressive)
#   pitch_range      → max - min pitch (how wide the pitch swings)
#   pitch_stability  → 1 - coeff_of_variation (higher = more stable)
#   jitter           → cycle-to-cycle pitch irregularity (voice tremor)
#   shimmer          → cycle-to-cycle amplitude irregularity (hoarseness)
#   hnr              → Harmonics-to-Noise Ratio (voice clarity, dB)
#   energy_mean      → average RMS volume of speech
#   energy_std       → variation in loudness
#   energy_range     → loudest - quietest moment
#   zcr_mean         → zero crossing rate (measures breathiness)
#   silence_ratio    → proportion of near-silent frames
#   duration         → length of the audio clip in seconds
#
# extract_prosodic():
#   - Loads audio at 16kHz mono
#   - Uses parselmouth (Praat) for pitch, jitter, shimmer, HNR
#   - Uses librosa for energy (RMS) and zero crossing rate
#   - Returns None if any error occurs (bad file), filtered later
#
# After extraction:
#   - All prosodic columns are StandardScaler normalized
#     (mean=0, std=1) so they don't dominate the mel features
# NOTE: This step takes ~15 minutes for ~8000+ files
# ============================================================
PROSODIC_COLS = ['pitch_mean','pitch_std','pitch_range','pitch_stability','jitter','shimmer','hnr','energy_mean','energy_std','energy_range','zcr_mean','silence_ratio','duration']

def extract_prosodic(filepath, sr=16000):
    try:
        y, _ = librosa.load(filepath, sr=sr, mono=True)
        if len(y) < sr: y = np.pad(y, (0, sr - len(y)))
        feats = {}
        snd = parselmouth.Sound(y, sampling_frequency=sr)
        pitch = call(snd, "To Pitch", 0.0, 75, 600)
        pv = pitch.selected_array['frequency']
        pv = pv[pv > 0]
        if len(pv) > 0:
            feats['pitch_mean']  = float(np.mean(pv))
            feats['pitch_std']   = float(np.std(pv))
            feats['pitch_range'] = float(np.ptp(pv))
            cv = np.std(pv) / np.mean(pv) if np.mean(pv) > 0 else 1.0
            feats['pitch_stability'] = float(np.clip(1.0 - cv, 0, 1))
        else:
            feats.update({'pitch_mean':0,'pitch_std':0,'pitch_range':0,'pitch_stability':0.5})
        
        pp = call(snd, "To PointProcess (periodic, cc)", 75, 600)
        try:
            feats['jitter']  = float(call(pp, "Get jitter (local)", 0,0,0.0001,0.02,1.3))
            feats['shimmer'] = float(call([snd,pp], "Get shimmer (local)", 0,0,0.0001,0.02,1.3,1.6))
        except:
            feats['jitter']  = 0.05
            feats['shimmer'] = 0.10
            
        try:
            hn = call(snd, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
            feats['hnr'] = float(call(hn, "Get mean", 0, 0))
        except: feats['hnr'] = 5.0
        
        rms = librosa.feature.rms(y=y)[0]
        feats['energy_mean']  = float(np.mean(rms))
        feats['energy_std']   = float(np.std(rms))
        feats['energy_range'] = float(np.ptp(rms))
        feats['zcr_mean'] = float(np.mean(librosa.feature.zero_crossing_rate(y)[0]))
        
        thr = np.max(np.abs(y)) * 0.02
        feats['silence_ratio'] = float(np.mean(np.abs(y) < thr))
        feats['duration'] = float(len(y) / sr)
        return feats
    except Exception as e: return None

print("⏳ Extracting prosodic features (This takes ~15 minutes)...")
prosodic_records = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    f = extract_prosodic(row['filepath'])
    if f: f['_idx'] = idx; prosodic_records.append(f)

pro_df = pd.DataFrame(prosodic_records).set_index('_idx')
df = df.join(pro_df).dropna(subset=PROSODIC_COLS).reset_index(drop=True)
scaler = StandardScaler()
df[PROSODIC_COLS] = scaler.fit_transform(df[PROSODIC_COLS])
print(f"✅ Extracting complete! Remaining files: {len(df)}")


## CNN + BiLSTM Architecture (Part 2)

In [ ]:
# ============================================================
# CELL 7: Define the Dataset class and the Neural Network model
#
# === AudioDataset ===
#   PyTorch Dataset that loads one audio file at a time:
#   1. Load .wav from disk using torchaudio
#   2. Convert to mono if stereo
#   3. Resample to 16000 Hz if needed
#   4. Pad/trim to exactly 3 seconds (48000 samples)
#   5. Convert to MelSpectrogram (128 mel bins, 1024 FFT)
#   6. Convert amplitude to dB scale (AmplitudeToDB)
#   7. Normalize mel: (mel - mean) / std  → zero mean, unit variance
#   8. Pad/trim time axis to MAX_FRAMES=94 (consistent tensor size)
#   9. Pack the prosodic vector (13 values) as a float tensor
#   Returns: dict with mel, prosodic, label, and 4 score targets
#
# === ConvBlock ===
#   A reusable CNN building block:
#   Conv2d → BN → ReLU → Conv2d → BN → ReLU → MaxPool → Dropout2d
#   Two conv layers per block capture local patterns before pooling
#
# === AudioConfidenceModel (main model) ===
#   Input: mel spectrogram (1 x 128 x 94) + prosodic vector (13,)
#
#   [CNN Branch]
#   4 ConvBlocks progressively extract spatial features from the mel:
#     Block1: 1→32ch, pool (2,2)     – detects basic frequency patterns
#     Block2: 32→64ch, pool (2,2)    – detects mid-level spectral patterns
#     Block3: 64→128ch, pool (2,1)   – focuses on frequency, keeps time
#     Block4: 128→128ch, pool (2,1)  – further refines frequency features
#   Output reshapes to (Batch, 23 time-steps, 1024 features)
#   cnn_proj: Linear(1024→256) + LayerNorm + ReLU → compress features
#
#   [BiLSTM Branch]
#   2-layer Bidirectional LSTM (hidden=256, bidirectional → output=512)
#   Processes the 23 time-steps sequentially in both directions
#   Captures temporal patterns: rising pitch, trailing off, hesitations
#
#   [Attention]
#   Learns which of the 23 time-steps matter most
#   Softmax weights → weighted sum of LSTM outputs → single 512-d vector
#
#   [Prosodic Encoder]
#   Linear(13→64) + ReLU + BatchNorm1d → encodes the 13 prosodic features
#
#   [Fusion]
#   Concat [pooled(512) + prosodic(64)] → Linear(576→256) → Linear(256→128)
#   Combines acoustic + linguistic signals into one shared representation
#
#   [Output Heads] – all share the 128-d fused embedding:
#     head_emotion   → Linear → 6 logits (CrossEntropy with labels)
#     head_confidence→ Linear → 1 score (0-1, via Sigmoid)
#     head_pitch     → Linear → 1 score (0-1, via Sigmoid)
#     head_fluency   → Linear → 1 score (0-1, via Sigmoid)
#     head_energy    → Linear → 1 score (0-1, via Sigmoid)
# ============================================================
MAX_FRAMES = 94
PROSODIC_DIM = 13
NUM_EMOTIONS = 6

class AudioDataset(Dataset):
    def __init__(self, dataframe, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment
        self.mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=128)
        self.db_transform = T.AmplitudeToDB()

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, sr = torchaudio.load(row['filepath'])
        if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
        if sr != 16000: wav = T.Resample(sr, 16000)(wav)
        target_len = 16000 * 3
        if wav.shape[1] >= target_len: wav = wav[:, :target_len]
        else: wav = F.pad(wav, (0, target_len - wav.shape[1]))
        
        mel = self.db_transform(self.mel_transform(wav))
        mel = (mel - mel.mean()) / (mel.std() + 1e-9)
        if mel.shape[2] >= MAX_FRAMES: mel = mel[:, :, :MAX_FRAMES]
        else: mel = F.pad(mel, (0, MAX_FRAMES - mel.shape[2]))
        
        prosodic = torch.tensor([float(row[c]) for c in PROSODIC_COLS], dtype=torch.float32)
        return {
            'mel': mel.float(), 'prosodic': prosodic,
            'label': torch.tensor(int(row['label']), dtype=torch.long),
            'confidence_score': torch.tensor(float(row['confidence_score']), dtype=torch.float32),
            'pitch_score': torch.tensor(float(row['pitch_score']), dtype=torch.float32),
            'fluency_score': torch.tensor(float(row['fluency_score']), dtype=torch.float32),
            'energy_score': torch.tensor(float(row['energy_score']), dtype=torch.float32)
        }

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2,2)):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.MaxPool2d(pool), nn.Dropout2d(0.2)
        )
    def forward(self, x): return self.conv(x)

class AudioConfidenceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            ConvBlock(1, 32, (2,2)), ConvBlock(32, 64, (2,2)),
            ConvBlock(64, 128, (2,1)), ConvBlock(128, 128, (2,1))
        )
        self.cnn_proj = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256), nn.ReLU())
        self.bilstm = nn.LSTM(256, 256, 2, batch_first=True, bidirectional=True, dropout=0.3)
        self.attention = nn.Sequential(nn.Linear(512, 64), nn.Tanh(), nn.Linear(64, 1))
        self.prosodic_encoder = nn.Sequential(nn.Linear(13, 64), nn.ReLU(), nn.BatchNorm1d(64))
        self.fusion = nn.Sequential(nn.Linear(512 + 64, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 128), nn.ReLU())
        
        head = lambda out: nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, out))
        self.head_emotion = head(6)
        self.head_confidence = nn.Sequential(head(1), nn.Sigmoid())
        self.head_pitch = nn.Sequential(head(1), nn.Sigmoid())
        self.head_fluency = nn.Sequential(head(1), nn.Sigmoid())
        self.head_energy = nn.Sequential(head(1), nn.Sigmoid())

    def forward(self, mel, prosodic):
        B = mel.shape[0]
        x = self.cnn(mel).permute(0, 3, 1, 2).reshape(B, 23, -1)
        x = self.cnn_proj(x)
        lstm_out, _ = self.bilstm(x)
        weights = torch.softmax(self.attention(lstm_out), dim=1)
        pooled = (lstm_out * weights).sum(dim=1)
        pros = self.prosodic_encoder(prosodic)
        shared = self.fusion(torch.cat([pooled, pros], dim=1))
        
        return {
            'emotion': self.head_emotion(shared),
            'confidence': self.head_confidence(shared).squeeze(-1),
            'pitch': self.head_pitch(shared).squeeze(-1),
            'fluency': self.head_fluency(shared).squeeze(-1),
            'energy': self.head_energy(shared).squeeze(-1)
        }

model = AudioConfidenceModel().to(device)
print("✅ CNN+BiLSTM Model Assembled!")


## Training the Model (Part 3)

In [ ]:
# ============================================================
# CELL 8: Define loss function, create data loaders, and train the model
#
# === MultiTaskLoss ===
#   Handles 5 simultaneous learning objectives:
#     l_em   → CrossEntropyLoss on emotion prediction (classification)
#     l_conf → MSELoss on confidence score (regression)
#     l_pit  → MSELoss on pitch stability score (regression)
#     l_flu  → MSELoss on fluency score (regression)
#     l_eng  → MSELoss on energy score (regression)
#
#   Uses Learned Uncertainty Weighting (Kendall et al., 2018):
#     log_sigma is a trainable parameter per task
#     w(loss, i) = exp(-2*sigma_i) * loss + sigma_i
#     This lets the model automatically balance task weights during training
#     Tasks where it's confident → lower sigma → higher weight
#
# === Data Loaders ===
#   train_test_split: 85% train, 15% validation, stratified by emotion label
#   DataLoader: batches of 32, shuffled for training, sequential for val
#
# === Optimizer ===
#   AdamW optimizer with lr=1e-3
#   Optimizes both model.parameters() AND criterion.parameters()
#   (so log_sigma weights are also learned during training)
#
# === Training Loop (15 epochs) ===
#   Each epoch:
#   1. Set model to train mode (enables dropout)
#   2. For each batch: zero gradients → forward pass → compute loss
#      → backprop → update weights
#   3. Track average epoch loss
#   4. If this epoch's average loss < best loss seen so far:
#      → Save model weights + label encoder to /content/best_model.pth
#   Only the BEST model checkpoint is kept (not every epoch)
# ============================================================
class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(5))
        self.ce = nn.CrossEntropyLoss()
        self.mse = nn.MSELoss()

    def forward(self, preds, targets):
        l_em = self.ce(preds['emotion'], targets['label'])
        l_conf = self.mse(preds['confidence'], targets['confidence_score'])
        l_pit = self.mse(preds['pitch'], targets['pitch_score'])
        l_flu = self.mse(preds['fluency'], targets['fluency_score'])
        l_eng = self.mse(preds['energy'], targets['energy_score'])
        
        def w(loss, i): return torch.exp(-2 * self.log_sigma[i]) * loss + self.log_sigma[i]
        total = w(l_em, 0) + w(l_conf, 1) + w(l_pit, 2) + w(l_flu, 3) + w(l_eng, 4)
        return total

train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'])
train_loader = DataLoader(AudioDataset(train_df), batch_size=32, shuffle=True)
val_loader = DataLoader(AudioDataset(val_df), batch_size=32)

criterion = MultiTaskLoss().to(device)
optimizer = optim.AdamW(list(model.parameters()) + list(criterion.parameters()), lr=1e-3)

NUM_EPOCHS = 15
best_loss = float('inf')
SAVE_PATH = '/content/best_model.pth'

print("🚀 Starting Model Training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        mel = batch['mel'].to(device)
        prosodic = batch['prosodic'].to(device)
        targets = {k: batch[k].to(device) for k in ['label', 'confidence_score', 'pitch_score', 'fluency_score', 'energy_score']}
        
        optimizer.zero_grad()
        preds = model(mel, prosodic)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'model': model.state_dict(), 
            'label_encoder': label_encoder.classes_.tolist()
        }, SAVE_PATH)
        print(f"💾 Saved Best Model (Loss: {best_loss:.4f})")

print("✅ Training Complete!")


## ✅ Final Step: Download your Model!

In [ ]:
# ============================================================
# CELL 9: Download the trained model from Colab to your local machine
#
# - files.download() triggers a browser download of best_model.pth
# - best_model.pth contains:
#     'model'         → the trained model weights (state_dict)
#     'label_encoder' → list of emotion class names in order
#
# - After downloading, move best_model.pth into your dl/api/ folder
#   so the Flask/FastAPI backend can load and serve predictions
#
# - If the auto-download fails (common in some browsers):
#   → Look in the Colab left sidebar (Files icon)
#   → Right-click best_model.pth → Download
# ============================================================
from google.colab import files

try:
    files.download('/content/best_model.pth')
    print("Downloading your model! Move it into your dl/api/ folder.")
except Exception as e:
    print("Could not automatically download, please right click best_model.pth in the folder sidebar and click Download.")
